
# Análise dos cargos e remuneração dos servidores federais: Outubro de 2025

**Autor:** Rafael Theodoro Rocha

**Instituição:** PUC-Rio | Pós-Graduação em Data Science & Analytics

**Ano:** 2025

---

## ✅ Checklist do MVP (Requisitos do Projeto)

Este MVP de Engenharia de Dados segue a metodologia proposta na disciplina:

- [✅] **Objetivo do Trabalho:** Definição clara do problema e perguntas a responder.
- [✅] **Plataforma:** Utilização do **Databricks Free Edition**.
- [✅] **Fonte de Dados:** [portaldatransparencia.gov.br](portaldatransparencia.gov.br).
- [✅] **Coleta e Armazenamento:** Extração e armazenamento na nuvem.
- [✅] **Modelagem:** Construção de modelo de dados e Catálogo de Dados básico.
- [ ] **Carga:** Processo de ETL (Extração, Transformação e Carga) documentado.
- [ ] **Análise - Qualidade:** Análise da qualidade dos dados por atributo.
- [ ] **Análise - Solução:** Respostas às perguntas do objetivo, com discussão dos resultados.
- [ ] **Autoavaliação:** Discussão sobre objetivos atingidos, dificuldades e trabalhos futuros.

---


## 1. Objetivo do Trabalho

O objetivo principal deste MVP é realizar uma análise detalhada dos cargos e da remuneração dos servidores públicos federais civis do Poder Executivo, utilizando dados de Outubro de 2025 provenientes do Portal da Transparência. O escopo da análise exclui intencionalmente as carreiras militares e servidores do Banco Central (BACEN).

Para isso, será estruturado um pipeline completo em ambiente Databricks, abrangendo a ingestão dos múltiplos datasets, inspeção e limpeza inicial, tratamento e transformação dos dados. O processo culminará na criação de uma estrutura de dados (tabela tratada) que suporte a análise exploratória, o cálculo de métricas e a produção de visualizações, permitindo compreender a composição da força de trabalho e a dinâmica remuneratória no serviço público federal.

### Perguntas Principais:

1.  Qual o quantitativo total de servidores (ativos e inativos) no governo federal?
2.  Qual a distribuição percentual dos tipos de vínculo dos servidores na ativa (e.g., efetivos/concursados, comissionados, temporários)?
3.  Qual o quantitativo de servidores em situação de afastamento ou licença?
4.  Dos servidores afastados, quantos são por licença-saúde, licença para interesse/capacitação ou licença-prêmio?
5.  Qual o Órgão que possui o maior quantitativo de servidores?
6.  Qual a carreira/cargo com o maior quantitativo de servidores?
7.  Qual o Órgão com a maior remuneração média?
8.  Qual o cargo com a melhor remuneração média no governo federal?
9.  Quantas vagas existem em vacância no governo federal?
10. Quais os Órgãos que possuem o maior número absoluto de vagas em vacância?
11. Quais os cargos que possuem o maior número absoluto de vagas em vacância?
12. Qual a diferença na remuneração média entre servidores do sexo masculino e feminino?
13. Qual a idade média ou o tempo médio de serviço dos servidores públicos federais?


## 2. Plataforma

Este projeto foi desenvolvido utilizando a plataforma **Databricks Free Edition** para processamento e análise dos dados.


In [0]:
# Comando para verificar a versão do Spark no ambiente Databricks
print(f"Versão do Spark utilizada: {spark.version}")


## 3. Fonte de Dados

Os dados utilizados são dados extraídos do Portal da Transparência do Governo federal, disponíveis no endereço:

[portaldatransparencia.gov.br](portaldatransparencia.gov.br)

Foram utilizados os arquivos `.zip` do mês de referência de Outubro/2025:

*   `Servidores_SIAPE`
*   `Aposentados_SIAPE`


## 4. Coleta e Armazenamento

Nesta seção, realizaremos a ingestão dos dados brutos provenientes do Portal da Transparência. Os arquivos `.zip` (`Servidores_SIAPE` e `Aposentados_SIAPE`) serão descompactados e os CSVs resultantes serão carregados e armazenados em um local temporário na nuvem (e.g., DBFS/S3/ADLS), preparando o terreno para as etapas de tratamento subsequentes, conforme a metodologia do MVP.


### 4.1) Preparação do catálogo e schemas

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS servidores;

In [0]:
%sql
USE CATALOG servidores

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS staging;

CREATE SCHEMA IF NOT EXISTS bronze;

CREATE SCHEMA IF NOT EXISTS silver;

CREATE SCHEMA IF NOT EXISTS gold; 



###  4.2) Criação do volume para armazenamento e coleta dos dados 

In [0]:
%sql
USE CATALOG servidores;
USE SCHEMA staging;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dadosabertos 

In [0]:
dbutils.fs.cp(
    "https://portaldatransparencia.gov.br/download-de-dados/servidores/202510_Servidores_SIAPE",
    "dbfs:/Volumes/servidores/staging/dadosabertos"
)

dbutils.fs.cp(
    "https://portaldatransparencia.gov.br/download-de-dados/servidores/202510_Aposentados_SIAPE",
    "dbfs:/Volumes/servidores/staging/dadosabertos"
)

In [0]:
import zipfile

with zipfile.ZipFile(
    "/Volumes/servidores/staging/dadosabertos/202510_Aposentados_SIAPE", 
    "r"
) as zip_ref:
    zip_ref.extractall(
        "/Volumes/servidores/staging/dadosabertos/aposentados"
    )

with zipfile.ZipFile(
    "/Volumes/servidores/staging/dadosabertos/202510_Servidores_SIAPE", 
    "r"
) as zip_ref:
    zip_ref.extractall(
        "/Volumes/servidores/staging/dadosabertos/ativa"
    )

## 5. Modelagem

A etapa de modelagem visa estruturar os dados brutos de forma que facilitem a análise e a resposta às perguntas principais do MVP. Aqui será definido se a abordagem será um modelo *flat* ou um esquema estrela simplificado. Também será criado um **Catálogo de Dados**, descrevendo minimamente os atributos principais, domínios esperados, e a linhagem dos dados.


In [0]:
%sql
DROP TABLE IF EXISTS servidores_cadastro;
DROP TABLE IF EXISTS servidores_remuneracao;
DROP TABLE IF EXISTS aposentados_cadastro;
DROP TABLE IF EXISTS aposentados_remuneracao;

### 5.1) Camada bronze

A camada bronze foi responsável pela ingestão e armazenamento dos dados brutos extraídos do Portal da Transparência. Os arquivos CSV referentes aos servidores ativos e aposentados foram lidos e carregados em DataFrames Spark. Em seguida, os dados foram normalizados quanto aos nomes das colunas para garantir compatibilidade com o Delta Lake. Por fim, os DataFrames foram salvos como Delta Tables (`servidores_cadastro`, `servidores_remuneracao`, `aposentados_cadastro`, `aposentados_remuneracao`) na camada bronze, preservando a integridade e o histórico dos dados originais para futuras etapas de tratamento e análise.

In [0]:
%sql

USE CATALOG servidores;
USE SCHEMA bronze;



Os arquivos CSV extraídos dos arquivos ZIP foram lidos diretamente para DataFrames Spark, utilizando opções adequadas de cabeçalho, separador e encoding para garantir a correta formação e estruturação dos dados conforme o formato original.

In [0]:
df_servidores_cadastro = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/ativa/202510_Cadastro.csv")
display(df_servidores_cadastro.limit(5))

In [0]:
df_servidores_remuneracao = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/ativa/202510_Remuneracao.csv")
display(df_servidores_remuneracao.limit(5))


In [0]:
df_aposentados_cadastro = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/aposentados/202510_Cadastro.csv")
display(df_aposentados_cadastro.limit(5))

In [0]:

df_aposentados_remuneracao = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/aposentados/202510_Remuneracao.csv")
display(df_aposentados_remuneracao.limit(5))

Padronização dos nomes das colunas dos DataFrames para garantir compatibilidade com o Delta Lake

In [0]:
# Funções de normalização de nomes de colunas:
# - normalizar_nome_colunas: padroniza nomes removendo acentuação, espaços, caracteres especiais, substitui R$ por BRL e U$ por USD, limita a 64 caracteres e ajusta termos específicos.
# - colunas_a_normalizar: retorna pares (original, normalizado) para colunas que serão modificadas.

import unicodedata
import re

def normalizar_nome_colunas(col):
    col = col.replace('R$', 'BRL').replace('U$', 'USD')
    col = col.replace('$', '')
    col = unicodedata.normalize('NFKD', col).encode('ASCII', 'ignore').decode()
    col = re.sub(r'[ ,;{}()\n\t=*]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    col = col.replace('VERBAS_INDENIZATORIAS', 'INDENIZACAO')
    return col[:64]

def colunas_a_normalizar(df):
    return [
        (col, normalizar_nome_colunas(col))
        for col in df.columns
        if normalizar_nome_colunas(col) != col
    ]



Verificação dos DataFrames cujos nomes de colunas serão normalizados

In [0]:
colunas_a_normalizar(df_servidores_cadastro)

In [0]:
colunas_a_normalizar(df_aposentados_cadastro)

In [0]:
colunas_a_normalizar(df_servidores_remuneracao)

In [0]:
colunas_a_normalizar(df_aposentados_remuneracao)

### Cadastro dos Servidores Ativos

Os dados cadastrais dos servidores ativos foram armazenados em uma Delta Table sem necessidade de normalização dos nomes das colunas.

In [0]:
# Criação da Delta Table 
df_servidores_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.servidores_cadastro")

In [0]:
%sql
select * from servidores_cadastro limit 5

### Cadastro dos Aposentados

Os dados cadastrais dos aposentados foram armazenados em uma Delta Table sem necessidade de normalização dos nomes das colunas.

In [0]:
# Criação da Delta Table 
df_aposentados_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.aposentados_cadastro")

In [0]:
%sql
select * from aposentados_cadastro limit 5

### Remuneração dos Servidores Ativos

Os dados de remuneração dos servidores ativos foram normalizados e armazenados em uma Delta Table


In [0]:
# Padronização dos nomes das colunas 
new_columns_servidores_remuneracao = [normalizar_nome_colunas(col) for col in df_servidores_remuneracao.columns]
df_servidores_remuneracao = df_servidores_remuneracao.toDF(*new_columns_servidores_remuneracao)

# Criação da Delta Table 
df_servidores_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.servidores_remuneracao")

In [0]:
%sql
select * from servidores_remuneracao limit 5

### Remuneração dos Servidores Aposentados

Os dados de remuneração dos servidores aposentados foram normalizados e armazenados em uma Delta Table

In [0]:

# Padronização dos nomes das colunas 
new_columns_aposentados_remumeracao = [normalizar_nome_colunas(col) for col in df_aposentados_remuneracao.columns]

df_aposentados_remuneracao = df_aposentados_remuneracao.toDF(*new_columns_aposentados_remumeracao)

# Criação da Delta Table 
df_aposentados_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.aposentados_remuneracao")


In [0]:
%sql
select * from aposentados_remuneracao limit 5

# 5.2) Camada Silver

Nesta etapa, aplicam-se técnicas de limpeza, seleção e padronização dos dados para garantir qualidade e relevância das informações.  
São removidas colunas sensíveis ou irrelevantes, tratados valores inválidos e realizadas conversões de tipos para facilitar análises e consultas futuras.

In [0]:
%sql
USE CATALOG servidores;
USE SCHEMA silver;

Cópia das tabelas da camada bronze para silver

In [0]:
%sql
-- Copia servidores_cadastro da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.servidores_cadastro AS
SELECT * FROM servidores.bronze.servidores_cadastro;

-- Copia aposentados_cadastro da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.aposentados_cadastro AS
SELECT * FROM servidores.bronze.aposentados_cadastro;

-- Copia servidores_remuneracao da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.servidores_remuneracao AS
SELECT * FROM servidores.bronze.servidores_remuneracao;

-- Copia aposentados_remuneracao da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.aposentados_remuneracao AS
SELECT * FROM servidores.bronze.aposentados_remuneracao;

Processo de análise, filtragem de colunas e tratamento de dados nas tabelas `servidores_cadastro`

In [0]:
%sql
select * from servidores.silver.servidores_cadastro limit 5

In [0]:

df_servidores_cadastro = spark.table("servidores.silver.servidores_cadastro")

display(df_servidores_cadastro.columns)

In [0]:
from pyspark.sql.functions import to_date

# Remoção de colunas que não agregam valor para análise:
# - Dados sensíveis (CPF, MATRÍCULA)
# - Detalhes excessivos de cargo, função, lotação e exercício que não são relevantes para agregações ou perguntas analíticas
# - Informações administrativas e documentos que não contribuem para as análises propostas
lista_colunas_remover = [
    'CPF', 'MATRICULA', 'CLASSE_CARGO', 'REFERENCIA_CARGO', 'PADRAO_CARGO', 'NIVEL_CARGO',
    'SIGLA_FUNCAO', 'NIVEL_FUNCAO', 'FUNCAO', "OPCAO_PARCIAL", 'COD_UORG_LOTACAO', 'UORG_LOTACAO',
    'COD_UORG_EXERCICIO', 'UORG_EXERCICIO', 'COD_ORG_EXERCICIO', 'ORG_EXERCICIO',
    'COD_ORGSUP_EXERCICIO', 'ORGSUP_EXERCICIO', 'REGIME_JURIDICO', 'JORNADA_DE_TRABALHO',
    'DATA_INGRESSO_CARGOFUNCAO', 'DATA_NOMEACAO_CARGOFUNCAO', 'DATA_INGRESSO_ORGAO',
    'DOCUMENTO_INGRESSO_SERVICOPUBLICO', 'DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO',
    'DIPLOMA_INGRESSO_CARGOFUNCAO', 'DIPLOMA_INGRESSO_ORGAO', 'DIPLOMA_INGRESSO_SERVICOPUBLICO'
]

df_filtrado_servidores_cadastro = df_servidores_cadastro.drop(*lista_colunas_remover)

# Substituição de valores inválidos ou faltantes por None para padronização e futura análise de qualidade dos dados
df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.replace("-1", None)
df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.replace(
    ["Sem informaç", "Inválido", "Sem informação", ""],
    None,
    subset=["DESCRICAO_CARGO", "ORGSUP_LOTACAO", "ATIVIDADE"]
)

# Transformação das colunas de data de afastamento para tipo date:
# - Mantidas mesmo se todos os valores forem nulos, pois são relevantes para avaliação da qualidade dos dados
#   e para responder perguntas sobre ausência de registros ou padrões de afastamento

df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.withColumn(
    "DATA_INICIO_AFASTAMENTO", 
    to_date("DATA_INICIO_AFASTAMENTO", "yyyy-MM-dd")
).withColumn(
    "DATA_TERMINO_AFASTAMENTO", 
    to_date("DATA_TERMINO_AFASTAMENTO", "yyyy-MM-dd")
)

display(df_filtrado_servidores_cadastro.limit(5))

In [0]:
# Criação da Delta Table 

df_filtrado_servidores_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.servidores_cadastro")

In [0]:
%sql
-- Comentário para a tabela servidores_cadastro
COMMENT ON TABLE servidores_cadastro IS 'Tabela de dados cadastrais dos servidores ativos, com informações limpas, padronizadas e prontas para análises de vínculos, cargos, atividades e órgãos de lotação.';

-- Comentários para o catálogo de dados da tabela servidores_cadastro
COMMENT ON COLUMN servidores_cadastro.Id_SERVIDOR_PORTAL IS 'Identificador único do servidor no portal';
COMMENT ON COLUMN servidores_cadastro.NOME IS 'Nome completo do servidor';
COMMENT ON COLUMN servidores_cadastro.DESCRICAO_CARGO IS 'Descrição do cargo ocupado pelo servidor';
COMMENT ON COLUMN servidores_cadastro.CODIGO_ATIVIDADE IS 'Código da atividade exercida pelo servidor';
COMMENT ON COLUMN servidores_cadastro.ATIVIDADE IS 'Descrição da atividade exercida pelo servidor';
COMMENT ON COLUMN servidores_cadastro.COD_ORG_LOTACAO IS 'Código do órgão de lotação do servidor';
COMMENT ON COLUMN servidores_cadastro.ORG_LOTACAO IS 'Nome do órgão de lotação do servidor';
COMMENT ON COLUMN servidores_cadastro.COD_ORGSUP_LOTACAO IS 'Código do órgão superior de lotação';
COMMENT ON COLUMN servidores_cadastro.ORGSUP_LOTACAO IS 'Nome do órgão superior de lotação';
COMMENT ON COLUMN servidores_cadastro.COD_TIPO_VINCULO IS 'Código do tipo de vínculo do servidor';
COMMENT ON COLUMN servidores_cadastro.TIPO_VINCULO IS 'Descrição do tipo de vínculo do servidor';
COMMENT ON COLUMN servidores_cadastro.SITUACAO_VINCULO IS 'Situação atual do vínculo do servidor';
COMMENT ON COLUMN servidores_cadastro.DATA_INICIO_AFASTAMENTO IS 'Data de início do afastamento do servidor, formato yyyy-MM-dd, NULL quando não aplicável';
COMMENT ON COLUMN servidores_cadastro.DATA_TERMINO_AFASTAMENTO IS 'Data de término do afastamento do servidor, formato yyyy-MM-dd, NULL quando não aplicável';
COMMENT ON COLUMN servidores_cadastro.UF_EXERCICIO IS 'Unidade Federativa de exercício do servidor';

DESCRIBE servidores_cadastro;

In [0]:
%sql
DESCRIBE DETAIL servidores_cadastro;


Processos de filtragem e transformação aplicados a tabela `aposentados_cadastro`.

In [0]:
%sql
select * from servidores.silver.aposentados_cadastro limit 5
    

In [0]:
df_aposentados_cadastro = spark.table("servidores.silver.aposentados_cadastro")

print(df_aposentados_cadastro.columns)

In [0]:
from pyspark.sql.functions import try_to_date

# Remoção de colunas que não agregam valor para análise:
# - Dados sensíveis (CPF, MATRÍCULA)
# - Detalhes administrativos e documentos que não contribuem para as análises propostas
lista_colunas_remover = [
    'CPF', 'MATRICULA', 'COD_UORG_LOTACAO', 'UORG_LOTACAO', 'REGIME_JURIDICO',
    'JORNADA_DE_TRABALHO', 'DATA_INGRESSO_CARGOFUNCAO', 'DATA_NOMEACAO_CARGOFUNCAO',
    'DATA_INGRESSO_ORGAO', 'DOCUMENTO_INGRESSO_SERVICOPUBLICO',
    'DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO', 'DIPLOMA_INGRESSO_CARGOFUNCAO',
    'DIPLOMA_INGRESSO_ORGAO', 'DIPLOMA_INGRESSO_SERVICOPUBLICO'
]

df_filtrado_aposentados_cadastro = df_aposentados_cadastro.drop(*lista_colunas_remover)

# Substituição de valores inválidos ou faltantes por None para padronização e futura análise de qualidade dos dados
df_filtrado_aposentados_cadastro = df_filtrado_aposentados_cadastro.replace("-1", None)
df_filtrado_aposentados_cadastro = df_filtrado_aposentados_cadastro.replace(
    ["Sem informaç", "Inválido", "Sem informação", ""],
    None,
    subset=["DESCRICAO_CARGO", "ORGSUP_LOTACAO"]
)

# Transformação da coluna de data de aposentadoria para tipo date:
# - Mantida mesmo se todos os valores forem nulos, pois é relevante para avaliação da qualidade dos dados
df_filtrado_aposentados_cadastro = df_filtrado_aposentados_cadastro.withColumn(
    "DATA_APOSENTADORIA", 
    try_to_date("DATA_APOSENTADORIA", "dd-MM-yyyy")
)


display(df_filtrado_aposentados_cadastro.limit(5))

In [0]:
df_filtrado_aposentados_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.aposentados_cadastro")

In [0]:
%sql
-- Comentário para a tabela aposentados_cadastro
COMMENT ON TABLE aposentados_cadastro IS 'Tabela de dados cadastrais dos servidores aposentados, com informações limpas, padronizadas e prontas para análises de vínculos, cargos, tipos de aposentadoria e órgãos de lotação.';

-- Comentários para o catálogo de dados da tabela aposentados_cadastro
COMMENT ON COLUMN aposentados_cadastro.Id_SERVIDOR_PORTAL IS 'Identificador único do servidor no portal';
COMMENT ON COLUMN aposentados_cadastro.NOME IS 'Nome completo do servidor aposentado';
COMMENT ON COLUMN aposentados_cadastro.COD_TIPO_APOSENTADORIA IS 'Código do tipo de aposentadoria';
COMMENT ON COLUMN aposentados_cadastro.TIPO_APOSENTADORIA IS 'Descrição do tipo de aposentadoria';
COMMENT ON COLUMN aposentados_cadastro.DATA_APOSENTADORIA IS 'Data da aposentadoria, formato yyyy-MM-dd';
COMMENT ON COLUMN aposentados_cadastro.DESCRICAO_CARGO IS 'Descrição do cargo ocupado pelo servidor antes da aposentadoria';
COMMENT ON COLUMN aposentados_cadastro.COD_ORG_LOTACAO IS 'Código do órgão de lotação do servidor';
COMMENT ON COLUMN aposentados_cadastro.ORG_LOTACAO IS 'Nome do órgão de lotação do servidor';
COMMENT ON COLUMN aposentados_cadastro.COD_ORGSUP_LOTACAO IS 'Código do órgão superior de lotação';
COMMENT ON COLUMN aposentados_cadastro.ORGSUP_LOTACAO IS 'Nome do órgão superior de lotação';
COMMENT ON COLUMN aposentados_cadastro.COD_TIPO_VINCULO IS 'Código do tipo de vínculo do servidor';
COMMENT ON COLUMN aposentados_cadastro.TIPO_VINCULO IS 'Descrição do tipo de vínculo do servidor';
COMMENT ON COLUMN aposentados_cadastro.SITUACAO_VINCULO IS 'Situação atual do vínculo do servidor';

DESCRIBE aposentados_cadastro;

In [0]:
%sql
DESCRIBE DETAIL aposentados_cadastro;


### Processos de análise, filtragem e transformação das colunas das tabelas `servidores_remuneracao` e `servidores_cadastro`


 Funções  utilizadas nos processos de filtragem e transformação das tabelas `servidores_remuneracao` e `aposentados_remuneracao`.

In [0]:
%python
def verificar_colunas_remuneracao_nao_vazias(df):
    """
    Retorna uma lista das colunas de remuneração (terminadas em _USD ou _BRL) que possuem valores válidos.
    Considera como válidos os valores diferentes de '0,00' e None.
    """
    colunas_remuneracao = [
        col for col in df.columns
        if (col.endswith('_USD') or col.endswith('_BRL'))
        and col not in ['ANO', 'MES', 'ID_SERVIDOR_PORTAL', 'NOME']
    ]
    resultado = []
    for col_name in colunas_remuneracao:
        valores_unicos = (
            df.select(col_name)
            .distinct()
            .rdd.flatMap(lambda x: x)
            .collect()
        )
        valores_filtrados = [valor for valor in valores_unicos if valor not in ['0,00', None]]
        if valores_filtrados:
            resultado.append(col_name)
    return resultado

def obter_colunas_remuneracao_vazias(df):
    """
    Retorna uma lista das colunas de remuneração (terminadas em _USD ou _BRL) que estão totalmente vazias,
    ou seja, preenchidas apenas com valores nulos ou '0,00'.
    """
    colunas_remuneracao = [
        col for col in df.columns
        if (col.endswith('_USD') or col.endswith('_BRL'))
        and col not in ['ANO', 'MES', 'ID_SERVIDOR_PORTAL', 'NOME']
    ]
    colunas_vazias = []
    for col in colunas_remuneracao:
        # Conta linhas não nulas e diferentes de '0,00'
        count_not_null = df.filter(
            (df[col].isNotNull()) & (df[col] != '0,00')
        ).count()
        if count_not_null == 0:
            colunas_vazias.append(col)
    return colunas_vazias

In [0]:
%python
from pyspark.sql.functions import regexp_replace, col

def converter_colunas_financeiras_para_double(df):
    """
    Converte colunas financeiras (terminadas em BRL ou USD) de string numérica com vírgula
    para tipo double, substituindo ',' por '.' antes do cast.
    Decisão: double é suficiente para agregações analíticas e oferece melhor performance
    quando precisão extrema não é necessária.
    """
    colunas_financeiras = [
        coluna for coluna in df.columns
        if coluna.endswith('BRL') or coluna.endswith('USD')
    ]
    for coluna in colunas_financeiras:
        df = df.withColumn(
            coluna,
            regexp_replace(col(coluna), ",", ".").cast("double")
        )
    return df

Processos de filtragem e transformação aplicados a tabela `servidores_remuneração`.

In [0]:
%sql
select * from servidores.silver.servidores_remuneracao limit 5

In [0]:
df_servidores_remuneracao = spark.table("servidores.silver.servidores_remuneracao")
display(df_servidores_remuneracao.columns)

In [0]:
# Remoção das colunas 'CPF' e de indenizações militares:
# 'CPF' é excluído por ser dado sensível e não relevante para análise agregada.
# Colunas de indenização militar são removidas por por não se aplicarem ao escopo da análise.

lista_inicial_col_remover = [ ] 
lista_inicial_col_remover = ['CPF','INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD']

In [0]:
# Identifica colunas de remuneração totalmente vazias usando funções Spark
lista_remuneracao_col_remover = [ ]

lista_remuneracao_col_remover = obter_colunas_remuneracao_vazias(df_servidores_remuneracao)

# Junta colunas a remover: sensíveis, militares e financeiras vazias
lista_final_remover_cols = [ ]
lista_final_remover_cols = lista_colunas_remover + lista_col_remover_remuneracao
print(lista_final_remover_cols)

In [0]:
df_filtrado_servidor_remuneracao = df_servidores_remuneracao.drop(*lista_final_remover_cols)

df_filtrado_servidor_remuneracao.printSchema()

In [0]:
# Converte colunas financeiras para tipo double 
df_final_servidor_remuneracao = converter_colunas_financeiras_para_double(df_filtrado_servidor_remuneracao)
df_final_servidor_remuneracao.printSchema()

In [0]:
%python

# Cria tabela Delta tratada e padronizada para remuneração dos servidores
df_final_servidor_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.servidores_remuneracao")



In [0]:
%sql
COMMENT ON TABLE servidores_remuneracao IS 'Tabela de dados de remuneração dos servidores, com informações limpas, padronizadas e prontas para análises de valores, tipos de remuneração, deduções e indenizações.';

COMMENT ON COLUMN servidores_remuneracao.ANO IS 'Ano de referência da remuneração';
COMMENT ON COLUMN servidores_remuneracao.MES IS 'Mês de referência da remuneração';
COMMENT ON COLUMN servidores_remuneracao.Id_SERVIDOR_PORTAL IS 'Identificador único do servidor no portal';
COMMENT ON COLUMN servidores_remuneracao.NOME IS 'Nome do servidor';
COMMENT ON COLUMN servidores_remuneracao.REMUNERACAO_BASICA_BRUTA_BRL IS 'Remuneração básica bruta em reais';
COMMENT ON COLUMN servidores_remuneracao.`ABATE-TETO_BRL` IS 'Valor abatido do teto constitucional em reais';
COMMENT ON COLUMN servidores_remuneracao.GRATIFICACAO_NATALINA_BRL IS 'Gratificação natalina em reais';
COMMENT ON COLUMN servidores_remuneracao.FERIAS_BRL IS 'Valor referente a férias em reais';
COMMENT ON COLUMN servidores_remuneracao.OUTRAS_REMUNERACOES_EVENTUAIS_BRL IS 'Outras remunerações eventuais em reais';
COMMENT ON COLUMN servidores_remuneracao.IRRF_BRL IS 'Imposto de Renda Retido na Fonte em reais';
COMMENT ON COLUMN servidores_remuneracao.`PSS/RPGS_BRL` IS 'Contribuição previdenciária (PSS/RPGS) em reais';
COMMENT ON COLUMN servidores_remuneracao.DEMAIS_DEDUCOES_BRL IS 'Demais deduções obrigatórias em reais';
COMMENT ON COLUMN servidores_remuneracao.TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL IS 'Taxa de ocupação de imóvel funcional em reais';
COMMENT ON COLUMN servidores_remuneracao.REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL IS 'Remuneração após deduções obrigatórias em reais';
COMMENT ON COLUMN servidores_remuneracao.`INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL` IS 'Indenizações registradas em sistemas de pessoal civil em reais';
COMMENT ON COLUMN servidores_remuneracao.`INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL` IS 'Indenização do Programa de Desligamento Voluntário MP 792/2017 em reais';
COMMENT ON COLUMN servidores_remuneracao.TOTAL_DE_INDENIZACAO_BRL IS 'Total de indenizações em reais';

In [0]:
%sql
DESCRIBE servidores_remuneracao

In [0]:
%sql
DESCRIBE DETAIL servidores_remuneracao

Processos de filtragem e transformação aplicados a tabela `aposentados_remuneração`.

In [0]:
%sql
select * from servidores.silver.aposentados_remuneracao limit 5

In [0]:
df_aposentados_remuneracao = spark.table("servidores.silver.aposentados_remuneracao")
display(df_aposentados_remuneracao.columns)

In [0]:
# Remoção das colunas 'CPF' e de indenizações militares:
# 'CPF' é excluído por ser dado sensível e não relevante para análise agregada.
# Colunas de indenização militar são removidas por por não se aplicarem ao escopo da análise.

lista_inicial_col_remover = [ ] 
lista_inicial_col_remover = ['CPF','INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD']

In [0]:
# Identifica colunas de remuneração totalmente vazias usando funções Spark
lista_remuneracao_col_remover = [ ]

lista_remuneracao_col_remover = obter_colunas_remuneracao_vazias(df_aposentados_remuneracao)

# Junta colunas a remover: sensíveis, militares e financeiras vazias
lista_final_remover_cols = [ ]
lista_final_remover_cols = lista_colunas_remover + lista_col_remover_remuneracao
print(lista_final_remover_cols)

In [0]:
df_filtrado_aposentados_remuneracao = df_aposentados_remuneracao.drop(*lista_final_remover_cols)

df_filtrado_aposentados_remuneracao.printSchema()

In [0]:
# Converte colunas financeiras para tipo double 
df_final_aposentados_remuneracao = converter_colunas_financeiras_para_double(df_filtrado_aposentados_remuneracao)
df_final_aposentados_remuneracao.printSchema()

In [0]:
%python
df_final_aposentados_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.aposentados_remuneracao")



In [0]:
%sql
COMMENT ON TABLE aposentados_remuneracao IS 'Tabela de dados de remuneração dos aposentados, com informações limpas, padronizadas e prontas para análises de valores, tipos de remuneração, deduções e indenizações.';

COMMENT ON COLUMN aposentados_remuneracao.ANO IS 'Ano de referência da remuneração';
COMMENT ON COLUMN aposentados_remuneracao.MES IS 'Mês de referência da remuneração';
COMMENT ON COLUMN aposentados_remuneracao.Id_SERVIDOR_PORTAL IS 'Identificador único do servidor no portal';
COMMENT ON COLUMN aposentados_remuneracao.NOME IS 'Nome do aposentado';
COMMENT ON COLUMN aposentados_remuneracao.REMUNERACAO_BASICA_BRUTA_BRL IS 'Remuneração básica bruta em reais';
COMMENT ON COLUMN aposentados_remuneracao.`ABATE-TETO_BRL` IS 'Valor abatido do teto constitucional em reais';
COMMENT ON COLUMN aposentados_remuneracao.GRATIFICACAO_NATALINA_BRL IS 'Gratificação natalina em reais';
COMMENT ON COLUMN aposentados_remuneracao.FERIAS_BRL IS 'Valor referente a férias em reais';
COMMENT ON COLUMN aposentados_remuneracao.OUTRAS_REMUNERACOES_EVENTUAIS_BRL IS 'Outras remunerações eventuais em reais';
COMMENT ON COLUMN aposentados_remuneracao.IRRF_BRL IS 'Imposto de Renda Retido na Fonte em reais';
COMMENT ON COLUMN aposentados_remuneracao.`PSS/RPGS_BRL` IS 'Contribuição previdenciária (PSS/RPGS) em reais';
COMMENT ON COLUMN aposentados_remuneracao.DEMAIS_DEDUCOES_BRL IS 'Demais deduções obrigatórias em reais';
COMMENT ON COLUMN aposentados_remuneracao.TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL IS 'Taxa de ocupação de imóvel funcional em reais';
COMMENT ON COLUMN aposentados_remuneracao.REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL IS 'Remuneração após deduções obrigatórias em reais';
COMMENT ON COLUMN aposentados_remuneracao.`INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL` IS 'Indenizações registradas em sistemas de pessoal civil em reais';
COMMENT ON COLUMN aposentados_remuneracao.`INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL` IS 'Indenização do Programa de Desligamento Voluntário MP 792/2017 em reais';
COMMENT ON COLUMN aposentados_remuneracao.TOTAL_DE_INDENIZACAO_BRL IS 'Total de indenizações em reais';

In [0]:
%sql
DESCRIBE aposentados_remuneracao

In [0]:
%sql
DESCRIBE DETAIL aposentados_remuneracao;

## 6. Carga

Nesta seção, os processos de Extração, Transformação e Carga (ETL) serão documentados e executados. Serão aplicadas as lógicas de limpeza, filtragem, enriquecimento e junção dos diferentes conjuntos de dados. A carga final ocorrerá na tabela tratada definida na seção de Modelagem, utilizando pipelines para garantir a reprodutibilidade do processo.


## 7. Análise - Qualidade

Uma análise crítica da qualidade de cada atributo do conjunto de dados tratado será realizada. Esta etapa verificará a presença de problemas como valores nulos, *outliers* inesperados, ou dados fora dos domínios esperados. Espera-se uma verificação dos valores por atributo para demonstrar a confiabilidade da base de análise.


## 8. Análise - Solução

Chegou o momento de solucionar o problema central do MVP e responder às perguntas delineadas na seção de Objetivo. Para cada pergunta, utilizaremos técnicas de análise exploratória e visualização para extrair *insights* dos dados tratados. Cada resposta técnica será seguida por uma discussão dos resultados, conectando os números obtidos ao contexto do serviço público federal.


## 9. Autoavaliação

Nesta seção, será feita uma discussão final sobre o atingimento dos objetivos propostos no início do trabalho. Abordaremos as dificuldades encontradas na execução do pipeline e da análise, as limitações do conjunto de dados e sugestões de trabalhos futuros para enriquecer a solução em um portfólio.
